# Migration: Dept_pkg Conversion
**Source File:** TARGET_ODI_SQL.sql
**Conversion Date:** 2024-05-22
**Description:** Migration of Oracle ODI staging and parameter logic to Databricks Delta Lake.

In [ ]:
dbutils.widgets.text("ODI_SESS_NO", "")
dbutils.widgets.text("DATASOURCE_NUM_ID", "")

### ETL Parameters

In [ ]:
%sql
-- SCEN_TASK_NO {1}: Get ETL last extract time parameter
CREATE OR REPLACE TEMPORARY VIEW v_dept_param AS
SELECT COALESCE(
    (SELECT date_format(last_run_dt, 'dd-MM-yy') 
     FROM workspace.odi_trg.log_table1 
     WHERE pkg_name = 'Dept_pkg'),
    date_format(current_timestamp(), 'dd-MM-yy')
) AS v_dept;

In [ ]:
v_dept = spark.sql("SELECT v_dept FROM v_dept_param").collect()[0][0]
print(f"V_Dept parameter: {v_dept}")

### Staging Table Preparation

In [ ]:
%sql
-- SCEN_TASK_NO {30}: Drop staging table
DROP TABLE IF EXISTS workspace.odi_trg.c_0filter;

In [ ]:
%sql
-- SCEN_TASK_NO {40}: Create staging table
CREATE TABLE workspace.odi_trg.c_0filter
(
    department_id   BIGINT,
    department_name STRING,
    manager_id      BIGINT,
    location_id     BIGINT,
    last_upd_dt     TIMESTAMP
)
USING DELTA;

### Data Loading

In [ ]:
%sql
-- SCEN_TASK_NO {50}: Load source data into staging
INSERT INTO workspace.odi_trg.c_0filter
(
    department_id,
    department_name,
    manager_id,
    location_id,
    last_upd_dt
)
SELECT
    src.department_id,
    src.department_name,
    src.manager_id,
    src.location_id,
    src.last_upd_dt
FROM workspace.odi_src.src_departments AS src
WHERE src.last_upd_dt >= (SELECT to_timestamp(v_dept, 'dd-MM-yy') FROM v_dept_param);

### Validation

In [ ]:
%sql
SELECT COUNT(*) AS row_count_staging FROM workspace.odi_trg.c_0filter;